Load into SQLite and clean with SQL

In [ ]:
import pandas as pd
import sqlite3

con = sqlite3.connect("saws.db")

df_all = pd.read_csv("saws_2021_2026.csv", parse_dates = ["date"])
df_all.to_sql("saws_raw", con, if_exists = "replace", index = False)

print("saws_raw loaded:", pd.read_sql("SELECT COUNT(*) FROM saws_raw", con).iloc[0, 0], "rows")

Pivot from long to wide 
(one row per station per date)

In [ ]:
con.executescript("""
                  DROP TABLE IF EXISTS saws_wide;
                  
                  CREATE TABLE saws_wide AS
                  SELECT
                    station_id,
                    station_name,
                    lat,
                    lon,
                    elevation_m,
                    date,
                    MAX(CASE WHEN measure = 'max_temp_c' THEN value END) AS max_temp_c,
                    MAX(CASE WHEN measure = 'min_temp_c' THEN value END) AS min_temp_c,
                    MAX(CASE WHEN measure = 'cloud_octas' THEN value END) AS cloud_octas
                  FROM saws_raw
                  GROUP BY station_id, station_name, lat, lon, elevation_m, date;
""")

pd.read_sql("SELECT * FROM saws_wide LIMIT 5", con)

Clean and produce the final SAWS table

In [ ]:
con.executescript("""
                  DROP TABLE IF EXISTS saws_clean;
                  
                  CREATE TABLE saws_clean AS
                  SELECT
                    station_id,
                    station_name,
                    lat,
                    lon,
                    elevation_m,
                    date,
                    
                    -- keep max and min 
                    ROUND(max_temp_c, 1) AS max_temp_c,
                    ROUND(min_temp_c, 1) AS min_temp_c,
                    
                    -- only compute range when both values exist AND max >= min
                    CASE 
                      WHEN max_temp_c IS NOT NULL AND min_temp_c IS NOT NULL AND max_temp_c >= min_temp_c
                      THEN ROUND(max_temp_c - min_temp_c, 1)
                      ELSE NULL
                    END AS temp_range_c,
                    
                    -- clip cloud to valid 0-8 octas scale
                    CASE
                        WHEN cloud_octas IS NULL THEN NULL
                        WHEN cloud_octas < 0 THEN 0.0
                        WHEN cloud_octas > 8 THEN 8.0
                        ELSE ROUND(cloud_octas, 2)
                    END AS cloud_octas,
                    
                    -- flag stations with no cloud data
                    CASE WHEN cloud_octas IS NULL THEN 0 ELSE 1 END AS has_cloud_data,
                    
                    -- flag impossible readings when max < min
                    CASE 
                        WHEN max_temp_c IS NOT NULL AND min_temp_c IS NOT NULL AND max_temp_c < min_temp_c
                        THEN 1 
                        ELSE 0
                    END AS temp_error_flag
                  FROM saws_wide;
""")

# check for temp error
pd.read_sql("""
            SELECT station_name, date, max_temp_c, min_temp_c
            FROM saws_clean
            WHERE temp_error_flag = 1
""", con)

Quick validation 

In [ ]:
pd.read_sql("""
            SELECT
              station_name,
              COUNT(*) AS total_days,
              SUM(CASE WHEN max_temp_c IS NULL THEN 1 ELSE 0 END) AS missing_max,
              SUM(CASE WHEN min_temp_c IS NULL THEN 1 ELSE 0 END) AS missing_min,
              SUM(CASE WHEN cloud_octas IS NULL THEN 1 ELSE 0 END) AS missing_cloud,
              SUM(temp_error_flag) AS temp_errors
            FROM saws_clean
            GROUP BY station_name
            ORDER BY lat ASC
""", con)

Pull the final clean table

In [ ]:
saws_clean = pd.read_sql("""
                         SELECT *
                         FROM saws_clean
                         ORDER BY station_id, date
""", con, parse_dates = ['date'])

con.close()

saws_clean.to_csv("saws_clean.csv", index = False)

In [ ]:
print(saws_clean.shape)
print(saws_clean.head())